In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# So we get same data every time we run
np.random.seed(42)
random.seed(42)

# Settings
NUM_TRANSACTIONS = 10000
NUM_CUSTOMERS = 500

# Indian cities for realism
cities = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", 
          "Pune", "Kolkata", "Ahmedabad", "Jaipur", "Surat"]

transaction_types = ["UPI", "NEFT", "IMPS", "Credit Card", "Debit Card"]

merchants = ["Amazon", "Flipkart", "Swiggy", "Zomato", "BigBasket",
             "Myntra", "Uber", "Ola", "BookMyShow", "Paytm Mall"]

# Generate data
records = []
start_date = datetime(2024, 1, 1)

for i in range(NUM_TRANSACTIONS):
    customer_id   = f"CUST_{random.randint(1, NUM_CUSTOMERS):04d}"
    transaction_id = f"TXN_{i+1:06d}"
    amount        = round(np.random.exponential(scale=3000), 2)
    city          = random.choice(cities)
    txn_type      = random.choice(transaction_types)
    merchant      = random.choice(merchants)
    txn_date      = start_date + timedelta(days=random.randint(0, 364),
                                           hours=random.randint(0, 23),
                                           minutes=random.randint(0, 59))
    
    # Fraud logic — high amount + odd hours = more likely fraud
    fraud_score = 0
    if amount > 15000:
        fraud_score += 0.4
    if txn_date.hour < 5 or txn_date.hour > 22:
        fraud_score += 0.3
    if txn_type == "IMPS":
        fraud_score += 0.1
    
    is_fraud = 1 if (fraud_score > 0.4 or random.random() < 0.01) else 0

    records.append({
        "transaction_id":   transaction_id,
        "customer_id":      customer_id,
        "amount":           amount,
        "city":             city,
        "transaction_type": txn_type,
        "merchant":         merchant,
        "transaction_date": txn_date,
        "hour_of_day":      txn_date.hour,
        "day_of_week":      txn_date.strftime("%A"),
        "is_fraud":         is_fraud
    })

# Convert to Pandas DataFrame
df_pandas = pd.DataFrame(records)

# Convert to Spark DataFrame and save as Delta table
df_spark = spark.createDataFrame(df_pandas)

df_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_transactions")

print(f"Done! Total rows: {df_pandas.shape[0]}")
print(f"Fraud transactions: {df_pandas['is_fraud'].sum()}")
print(f"Fraud rate: {round(df_pandas['is_fraud'].mean() * 100, 2)}%")

Done! Total rows: 10000
Fraud transactions: 106
Fraud rate: 1.06%


In [0]:
display(spark.sql("SELECT * FROM bronze_transactions LIMIT 10"))

transaction_id,customer_id,amount,city,transaction_type,merchant,transaction_date,hour_of_day,day_of_week,is_fraud
TXN_002501,CUST_0183,5272.4,Ahmedabad,Debit Card,Swiggy,2024-01-14T23:06:00.000Z,23,Sunday,0
TXN_002502,CUST_0293,4338.49,Mumbai,IMPS,Zomato,2024-03-29T05:14:00.000Z,5,Friday,0
TXN_002503,CUST_0376,2556.63,Bangalore,NEFT,Zomato,2024-04-11T04:18:00.000Z,4,Thursday,0
TXN_002504,CUST_0339,9373.91,Ahmedabad,NEFT,Paytm Mall,2024-03-09T01:07:00.000Z,1,Saturday,0
TXN_002505,CUST_0371,671.21,Hyderabad,IMPS,BigBasket,2024-12-17T13:09:00.000Z,13,Tuesday,0
TXN_002506,CUST_0427,347.12,Bangalore,Credit Card,Uber,2024-01-04T05:23:00.000Z,5,Thursday,0
TXN_002507,CUST_0090,5771.66,Kolkata,UPI,Flipkart,2024-03-21T06:04:00.000Z,6,Thursday,0
TXN_002508,CUST_0056,1734.9,Kolkata,NEFT,Swiggy,2024-05-16T22:33:00.000Z,22,Thursday,0
TXN_002509,CUST_0453,5631.34,Jaipur,Debit Card,Paytm Mall,2024-06-17T14:10:00.000Z,14,Monday,0
TXN_002510,CUST_0305,6707.31,Jaipur,Debit Card,Zomato,2024-08-17T13:11:00.000Z,13,Saturday,0


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Read from Bronze table
df_bronze = spark.read.format("delta").table("bronze_transactions")

print(f"Bronze row count: {df_bronze.count()}")

# ── Cleaning ──────────────────────────────────────────────────

# 1. Remove negative or zero amounts
df_silver = df_bronze.filter(F.col("amount") > 0)

# 2. Remove duplicate transaction IDs
df_silver = df_silver.dropDuplicates(["transaction_id"])

# 3. Ensure correct data types
df_silver = df_silver.withColumn("amount", F.col("amount").cast(DoubleType()))

# 4. Standardize city names (trim spaces, proper case)
df_silver = df_silver.withColumn("city", F.initcap(F.trim(F.col("city"))))

# ── New derived columns ───────────────────────────────────────

# 5. Is the transaction on a weekend?
df_silver = df_silver.withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin("Saturday", "Sunday"), 1).otherwise(0)
)

# 6. Amount category — used later in ML and dashboards
df_silver = df_silver.withColumn(
    "amount_category",
    F.when(F.col("amount") < 500,   "Low")
     .when(F.col("amount") < 5000,  "Medium")
     .when(F.col("amount") < 20000, "High")
     .otherwise("Very High")
)

# 7. Is it a late night transaction? (between 11pm and 5am)
df_silver = df_silver.withColumn(
    "is_late_night",
    F.when(
        (F.col("hour_of_day") >= 23) | (F.col("hour_of_day") <= 5), 1
    ).otherwise(0)
)

# 8. Add a processed timestamp so we know when this ran
df_silver = df_silver.withColumn(
    "processed_at", F.current_timestamp()
)

# ── Save as Silver Delta table ────────────────────────────────
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_transactions")

print(f"Silver row count: {df_silver.count()}")
print(f"New columns added: is_weekend, amount_category, is_late_night, processed_at")
print("Silver layer complete!")

Bronze row count: 10000
Silver row count: 10000
New columns added: is_weekend, amount_category, is_late_night, processed_at
Silver layer complete!


In [0]:
display(
    spark.sql("""
        SELECT 
            transaction_id,
            amount,
            amount_category,
            hour_of_day,
            is_late_night,
            day_of_week,
            is_weekend,
            is_fraud
        FROM silver_transactions
        ORDER BY is_fraud DESC
        LIMIT 10
    """)
)

transaction_id,amount,amount_category,hour_of_day,is_late_night,day_of_week,is_weekend,is_fraud
TXN_000437,226.64,Low,14,0,Thursday,0,1
TXN_001055,18546.58,High,4,1,Friday,0,1
TXN_000706,13803.6,High,7,0,Wednesday,0,1
TXN_000848,17304.24,High,3,1,Friday,0,1
TXN_000715,1287.13,Medium,16,0,Saturday,1,1
TXN_001105,15840.15,High,23,1,Saturday,1,1
TXN_001210,17554.59,High,23,1,Monday,0,1
TXN_000075,3681.29,Medium,20,0,Saturday,1,1
TXN_000471,36.69,Low,17,0,Wednesday,0,1
TXN_000084,197.0,Low,12,0,Monday,0,1


In [0]:
# ── Gold Table 1: Daily transaction summary ───────────────────
gold_daily = spark.sql("""
    SELECT
        DATE(transaction_date)        AS txn_date,
        COUNT(*)                      AS total_transactions,
        SUM(amount)                   AS total_amount,
        SUM(is_fraud)                 AS fraud_count,
        ROUND(SUM(is_fraud) * 100.0 
              / COUNT(*), 2)          AS fraud_rate_pct,
        ROUND(AVG(amount), 2)         AS avg_transaction_amount
    FROM silver_transactions
    GROUP BY DATE(transaction_date)
    ORDER BY txn_date
""")

gold_daily.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_daily_summary")

# ── Gold Table 2: Fraud by city ───────────────────────────────
gold_city = spark.sql("""
    SELECT
        city,
        COUNT(*)             AS total_transactions,
        SUM(is_fraud)        AS fraud_count,
        ROUND(SUM(is_fraud) * 100.0 
              / COUNT(*), 2) AS fraud_rate_pct,
        ROUND(AVG(amount),2) AS avg_amount
    FROM silver_transactions
    GROUP BY city
    ORDER BY fraud_rate_pct DESC
""")

gold_city.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_fraud_by_city")

# ── Gold Table 3: Fraud by transaction type ───────────────────
gold_txn_type = spark.sql("""
    SELECT
        transaction_type,
        COUNT(*)             AS total_transactions,
        SUM(is_fraud)        AS fraud_count,
        ROUND(SUM(is_fraud) * 100.0 
              / COUNT(*), 2) AS fraud_rate_pct
    FROM silver_transactions
    GROUP BY transaction_type
    ORDER BY fraud_rate_pct DESC
""")

gold_txn_type.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_fraud_by_type")

# ── Gold Table 4: High risk customers ────────────────────────
gold_customers = spark.sql("""
    SELECT
        customer_id,
        COUNT(*)             AS total_transactions,
        SUM(is_fraud)        AS fraud_count,
        ROUND(SUM(amount),2) AS total_amount_spent,
        ROUND(AVG(amount),2) AS avg_transaction_amount,
        MAX(is_late_night)   AS has_late_night_txn
    FROM silver_transactions
    GROUP BY customer_id
    HAVING SUM(is_fraud) > 0
    ORDER BY fraud_count DESC
    LIMIT 50
""")

gold_customers.write.format("delta").mode("overwrite") \
    .saveAsTable("gold_high_risk_customers")

print("Gold layer complete!")
print("Tables created:")
print("  - gold_daily_summary")
print("  - gold_fraud_by_city")
print("  - gold_fraud_by_type")
print("  - gold_high_risk_customers")

Gold layer complete!
Tables created:
  - gold_daily_summary
  - gold_fraud_by_city
  - gold_fraud_by_type
  - gold_high_risk_customers


In [0]:
display(spark.sql("SELECT * FROM gold_fraud_by_city"))

city,total_transactions,fraud_count,fraud_rate_pct,avg_amount
Hyderabad,954,13,1.36,3017.42
Chennai,1053,14,1.33,2857.34
Kolkata,983,13,1.32,3027.31
Pune,980,11,1.12,2722.46
Mumbai,999,11,1.10,2957.68
Ahmedabad,1009,10,0.99,3111.54
Surat,1034,10,0.97,3063.64
Bangalore,980,9,0.92,2751.63
Jaipur,996,9,0.90,2944.3
Delhi,1012,6,0.59,2868.11


In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 MB 132.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, 
                             confusion_matrix,
                             roc_auc_score)
import pandas as pd

# ── Load Silver table for training ───────────────────────────
df = spark.table("silver_transactions").toPandas()

# ── Feature engineering ───────────────────────────────────────
# Convert categorical columns to numbers
df["transaction_type_code"] = df["transaction_type"].astype("category").cat.codes
df["city_code"]             = df["city"].astype("category").cat.codes
df["merchant_code"]         = df["merchant"].astype("category").cat.codes

# Features we will train on
features = [
    "amount",
    "hour_of_day",
    "is_weekend",
    "is_late_night",
    "transaction_type_code",
    "city_code",
    "merchant_code"
]

X = df[features]
y = df["is_fraud"]

# ── Train/test split ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training rows : {len(X_train)}")
print(f"Testing rows  : {len(X_test)}")
print(f"Fraud in train: {y_train.sum()}")
print(f"Fraud in test : {y_test.sum()}")

# ── Train model inside MLflow experiment ──────────────────────
mlflow.set_experiment("/bankguard_fraud_detection")

with mlflow.start_run(run_name="xgboost_v1"):

    model = XGBClassifier(
        n_estimators    = 100,
        max_depth       = 4,
        learning_rate   = 0.1,
        scale_pos_weight= 10,  # handles class imbalance (few frauds)
        random_state    = 42,
        eval_metric     = "logloss",
        verbosity       = 0
    )

    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # ── Metrics ───────────────────────────────────────────────
    auc_score = roc_auc_score(y_test, y_proba)
    report    = classification_report(y_test, y_pred, output_dict=True)
    precision = report["1"]["precision"]
    recall    = report["1"]["recall"]
    f1        = report["1"]["f1-score"]

    # ── Log everything to MLflow ──────────────────────────────
    mlflow.log_param("n_estimators",     100)
    mlflow.log_param("max_depth",        4)
    mlflow.log_param("learning_rate",    0.1)
    mlflow.log_param("scale_pos_weight", 10)

    mlflow.log_metric("auc",       round(auc_score, 4))
    mlflow.log_metric("precision", round(precision, 4))
    mlflow.log_metric("recall",    round(recall,    4))
    mlflow.log_metric("f1_score",  round(f1,        4))

    # ── Save the model ────────────────────────────────────────
    mlflow.sklearn.log_model(model, "fraud_model")

    print("\n── Model Results ──────────────────────────────")
    print(f"AUC Score : {round(auc_score, 4)}")
    print(f"Precision : {round(precision, 4)}")
    print(f"Recall    : {round(recall,    4)}")
    print(f"F1 Score  : {round(f1,        4)}")
    print("\n── Confusion Matrix ───────────────────────────")
    print(confusion_matrix(y_test, y_pred))
    print("\nMLflow run complete — model saved!")

2026/05/19 11:35:40 INFO mlflow.tracking.fluent: Experiment with name '/bankguard_fraud_detection' does not exist. Creating a new experiment.


Training rows : 8000
Testing rows  : 2000
Fraud in train: 85
Fraud in test : 21


2026/05/19 11:35:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-3bff1d67-6d86.cloud.databricks.com/ml/experiments/2427473564745245/models/m-29bec9a1f6af4443b6b28d6fadf65048?o=7474655625103601
2026/05/19 11:35:45 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



── Model Results ──────────────────────────────
AUC Score : 0.5524
Precision : 0.4286
Recall    : 0.1429
F1 Score  : 0.2143

── Confusion Matrix ───────────────────────────
[[1975    4]
 [  18    3]]

MLflow run complete — model saved!


In [0]:
import mlflow

# Fetch the experiment
experiment = mlflow.get_experiment_by_name("/bankguard_fraud_detection")

# Get all runs
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

display(
    spark.createDataFrame(
        runs[["run_id", "status", 
              "metrics.auc", "metrics.precision", 
              "metrics.recall", "metrics.f1_score",
              "params.n_estimators", "params.learning_rate"]]
    )
)

run_id,status,metrics.auc,metrics.precision,metrics.recall,metrics.f1_score,params.n_estimators,params.learning_rate
b257fe9280664b5e91d4fb93cc6e24ee,FINISHED,0.5524,0.4286,0.1429,0.2143,100,0.1


In [0]:
from datetime import datetime

# ── Pull key metrics from Gold tables ─────────────────────────
daily_summary = spark.sql("""
    SELECT
        SUM(total_transactions) AS total_txns,
        ROUND(SUM(total_amount), 2) AS total_amount,
        SUM(fraud_count) AS total_fraud,
        ROUND(AVG(fraud_rate_pct), 2) AS avg_fraud_rate
    FROM gold_daily_summary
""").toPandas()

top_city = spark.sql("""
    SELECT city, fraud_count, fraud_rate_pct
    FROM gold_fraud_by_city
    ORDER BY fraud_rate_pct DESC
    LIMIT 1
""").toPandas()

top_txn_type = spark.sql("""
    SELECT transaction_type, fraud_count, fraud_rate_pct
    FROM gold_fraud_by_type
    ORDER BY fraud_rate_pct DESC
    LIMIT 1
""").toPandas()

top_customer = spark.sql("""
    SELECT customer_id, fraud_count, total_amount_spent
    FROM gold_high_risk_customers
    ORDER BY fraud_count DESC
    LIMIT 1
""").toPandas()

# ── Print the report ──────────────────────────────────────────
print("=" * 55)
print("       BANKGUARD — DAILY FRAUD REPORT")
print(f"       Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 55)

print(f"""
OVERALL SUMMARY
───────────────────────────────────────────────────────
Total Transactions  : {int(daily_summary['total_txns'][0]):,}
Total Amount        : ₹{daily_summary['total_amount'][0]:,.2f}
Total Fraud Cases   : {int(daily_summary['total_fraud'][0])}
Average Fraud Rate  : {daily_summary['avg_fraud_rate'][0]}%

HIGHEST RISK CITY
───────────────────────────────────────────────────────
City                : {top_city['city'][0]}
Fraud Cases         : {int(top_city['fraud_count'][0])}
Fraud Rate          : {top_city['fraud_rate_pct'][0]}%

RISKIEST TRANSACTION TYPE
───────────────────────────────────────────────────────
Type                : {top_txn_type['transaction_type'][0]}
Fraud Cases         : {int(top_txn_type['fraud_count'][0])}
Fraud Rate          : {top_txn_type['fraud_rate_pct'][0]}%

HIGHEST RISK CUSTOMER
───────────────────────────────────────────────────────
Customer ID         : {top_customer['customer_id'][0]}
Fraud Transactions  : {int(top_customer['fraud_count'][0])}
Total Spent         : ₹{top_customer['total_amount_spent'][0]:,.2f}
""")
print("=" * 55)
print("Report complete. All metrics sourced from Gold layer.")
print("=" * 55)

       BANKGUARD — DAILY FRAUD REPORT
       Generated: 2026-05-19 11:44

OVERALL SUMMARY
───────────────────────────────────────────────────────
Total Transactions  : 10,000
Total Amount        : ₹29,324,968.50
Total Fraud Cases   : 106
Average Fraud Rate  : 1.05%

HIGHEST RISK CITY
───────────────────────────────────────────────────────
City                : Hyderabad
Fraud Cases         : 13
Fraud Rate          : 1.36%

RISKIEST TRANSACTION TYPE
───────────────────────────────────────────────────────
Type                : IMPS
Fraud Cases         : 30
Fraud Rate          : 1.52%

HIGHEST RISK CUSTOMER
───────────────────────────────────────────────────────
Customer ID         : CUST_0006
Fraud Transactions  : 2
Total Spent         : ₹54,512.50

Report complete. All metrics sourced from Gold layer.
